# Notebook de Obtención, Limpieza y Transformación de Datos — Fase 2
## Proyecto: Análisis y Predicción de Diabetes

### ¿Qué es un pipeline de preprocesamiento?

Un pipeline de preprocesamiento corresponde a una secuencia estructurada de etapas mediante las cuales los datos en bruto son preparados para su posterior análisis y modelado. Su propósito es garantizar la calidad, consistencia y confiabilidad de la información antes de aplicar técnicas estadísticas o algoritmos de aprendizaje automático.

En este notebook se implementa el siguiente flujo de trabajo:

### Obtención → Exploración → Limpieza → Transformación → Escalamiento → Validación

| # | Etapa | Función principal |
|---|---|---|
| 0 | Configuración del entorno | — |
| 1 | Obtención y carga de datos | `cargar_datos()` |
| 2 | Exploración inicial (EDA) | `explorar_dataframe()` |
| 3 | Limpieza de datos | `eliminar_columnas()`, `castear_bloodpressure()`, `reemplazar_invalidos()`, `eliminar_duplicados()` |
| 4 | Imputación de valores faltantes | `imputar_mediana()` |
| 5 | Validación posterior a la limpieza | `validar_post_limpieza()` |
| 6 | Validación de rangos y outliers | `analizar_rangos()`, `detectar_outliers_iqr()` |
| 7 | Transformaciones y variables derivadas | `crear_variables_derivadas()`, `codificar_categoricas()` |
| 8 | Escalamiento de variables | `escalar_variables()` |
| 9 | Validación técnica final | `validar_dataset_final()` |
| 10 | Comparación antes / después | `resumen_comparativo()` |
| 11 | Exportación del dataset procesado | `exportar_dataset()` |

Cada etapa está encapsulada en una **función con un único propósito**, lo que favorece la reproducibilidad, la legibilidad y la reutilización del código.

> **Reproducibilidad:** ejecutar con `Kernel → Restart & Run All`.

## Introducción

La diabetes mellitus es una de las enfermedades crónicas más relevantes a nivel mundial y representa un importante desafío para los sistemas de salud debido a sus complicaciones asociadas y su creciente prevalencia. El análisis de datos clínicos permite identificar patrones y factores de riesgo que contribuyen al diagnóstico temprano y a la toma de decisiones informadas.

El conjunto de datos utilizado en este proyecto contiene información médica y demográfica de pacientes, incluyendo variables relacionadas con embarazos, niveles de glucosa, presión arterial, grosor de pliegues cutáneos, niveles de insulina, índice de masa corporal y antecedentes familiares de diabetes. A partir de estos atributos, se busca analizar la calidad de los datos y prepararlos adecuadamente para futuras etapas de modelado predictivo.

## Objetivo General (Fase 2)

Construir un pipeline de preprocesamiento reproducible que permita obtener un conjunto de datos limpio, consistente y preparado para las etapas posteriores de análisis exploratorio y modelado predictivo.

## Objetivos Específicos

* Cargar y explorar el conjunto de datos verificando su estructura, tipos de datos y calidad general.
* Identificar y tratar valores faltantes, inconsistencias, tipos incorrectos y posibles errores de registro.
* Detectar y eliminar filas duplicadas que puedan distorsionar el análisis.
* Detectar y analizar valores atípicos presentes en las variables numéricas.
* Transformar las variables cuando sea necesario para facilitar su interpretación y procesamiento.
* Generar variables derivadas que aporten valor analítico y clínico al dataset.
* Estandarizar las variables numéricas para asegurar escalas comparables entre atributos.
* Validar la integridad y consistencia del conjunto de datos resultante mediante verificaciones automáticas con `assert`.
* Generar una versión procesada del dataset que pueda ser utilizada en etapas posteriores de análisis y modelado.

## Descripción de los Atributos

El dataset contiene las siguientes variables:

<table>
    <thead><tr><th>Variable</th><th>Descripción</th></tr></thead>
    <tbody>
        <tr><td>Pregnancies</td><td>Número de embarazos registrados.</td></tr>
        <tr><td>Glucose</td><td>Concentración de glucosa en plasma (mg/dL).</td></tr>
        <tr><td>BloodPressure</td><td>Presión arterial diastólica (mm Hg).</td></tr>
        <tr><td>SkinThickness</td><td>Grosor del pliegue cutáneo del tríceps (mm).</td></tr>
        <tr><td>Insulin</td><td>Nivel de insulina sérica (mu U/ml).</td></tr>
        <tr><td>BMI</td><td>Índice de Masa Corporal (kg/m²).</td></tr>
        <tr><td>DiabetesPedigreeFunction</td><td>Indicador de predisposición genética a la diabetes.</td></tr>
        <tr><td>Age</td><td>Edad del paciente en años.</td></tr>
        <tr><td>Outcome</td><td>Variable objetivo: 1 indica presencia de diabetes y 0 indica ausencia.</td></tr>
    </tbody>
</table>

## Librerías Utilizadas

| Librería | Propósito |
|-----------|-----------|
| numpy | Operaciones numéricas, manejo de arreglos y generación de valores aleatorios para procesos reproducibles. |
| pandas | Carga, exploración y manipulación de datos mediante estructuras DataFrame. |
| matplotlib.pyplot | Generación de gráficos para el análisis exploratorio y la visualización de distribuciones de datos. |
| sklearn.preprocessing | Herramientas para la transformación y preparación de variables antes del modelado. |
| StandardScaler | Estandarización de variables numéricas, ajustando su media a 0 y su desviación estándar a 1. |

### Configuración del entorno

La instrucción **np.random.seed(42)** establece una semilla aleatoria fija para garantizar que cualquier proceso que involucre aleatoriedad produzca siempre los mismos resultados. Esto asegura la **reproducibilidad** del análisis y permite que otros investigadores obtengan los mismos resultados al ejecutar el notebook.

La instrucción **%matplotlib inline** permite que los gráficos generados con Matplotlib se visualicen directamente dentro del notebook, facilitando el análisis y la interpretación de los resultados.

## Resultado esperado de la Fase 2

Al finalizar este notebook, se dispondrá de una versión procesada del dataset denominada `diabetes_processed.csv`, libre de inconsistencias, con los valores tratados adecuadamente y preparada para las actividades de análisis exploratorio avanzado y construcción de modelos predictivos que se desarrollarán en las siguientes fases del proyecto.

## Importación de librerías

En esta sección se importan las librerías necesarias para la carga, exploración, transformación y visualización de los datos. Además, se configura el entorno para garantizar la reproducibilidad de los resultados y la visualización integrada de gráficos dentro del notebook.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Reproducibilidad
np.random.seed(42)
%matplotlib inline

# ── Importar funciones del pipeline desde src/utils.py ────────────────────
# Las funciones están centralizadas en src/utils.py para que sean
# reutilizables desde cualquier celda, sin depender del orden de ejecución.
sys.path.insert(0, os.path.abspath('..'))
from src.utils import *

print('✓ Funciones del pipeline importadas correctamente desde src/utils.py')

# ── Rutas ─────────────────────────────────────────────────────────────────
RUTA_RAW       = Path('../data/raw/diabetes_raw.csv')
RUTA_PROCESSED = Path('../data/processed/diabetes_processed.csv')
RUTA_PROCESSED.parent.mkdir(parents=True, exist_ok=True)

# Verificación de entorno
versiones = pd.DataFrame({
    'Componente': ['Python', 'NumPy', 'Pandas', 'Matplotlib', 'Scikit-Learn'],
    'Versión': [
        sys.version.split()[0],
        np.__version__,
        pd.__version__,
        matplotlib.__version__,
        __import__('sklearn').__version__
    ]
})
print(f'CSV encontrado : {RUTA_RAW.exists()}')
versiones

## Verificación del Entorno de Trabajo

Antes de comenzar el proceso de exploración y transformación de datos, es recomendable registrar las versiones de las principales librerías utilizadas en el proyecto. Esta información permite garantizar la reproducibilidad del análisis y facilita la identificación de posibles diferencias de comportamiento entre distintas versiones de las herramientas empleadas.

En esta sección se muestran las versiones de NumPy, Pandas, Matplotlib y Scikit-Learn utilizadas durante el desarrollo de la Fase 2.

In [ ]:
versiones = pd.DataFrame({
    "Componente": ["Python", "NumPy", "Pandas", "Matplotlib", "Scikit-Learn"],
    "Versión": [
        sys.version.split()[0],
        np.__version__,
        pd.__version__,
        matplotlib.__version__,
        __import__('sklearn').__version__
    ]
})
print(f"CSV encontrado : {RUTA_RAW.exists()}")
print(f"Ruta raw       : {RUTA_RAW.resolve()}")
versiones

## 1. Obtención de los datos

### ¿Qué se realiza en esta etapa?

El primer paso del proceso consiste en cargar el conjunto de datos desde un archivo CSV hacia un DataFrame de Pandas. El DataFrame será la estructura principal utilizada durante todo el análisis, ya que permite almacenar, consultar y transformar los datos de manera eficiente.

### ¿Por qué utilizar una función?

La lectura del archivo se encapsula dentro de una función denominada `cargar_datos(ruta)`. Este enfoque favorece la reutilización del código, mejora su organización y facilita el mantenimiento del proyecto. Además, permite centralizar el proceso de carga de datos en un único punto, evitando duplicar instrucciones en distintas partes del notebook.

La función incorpora un mecanismo de control de errores mediante la estructura `try-except`. De esta forma, si el archivo no existe o la ruta especificada es incorrecta, el programa informa al usuario mediante un mensaje claro y comprensible, evitando errores técnicos que puedan dificultar el diagnóstico del problema.

### Descripción del funcionamiento

La función recibe como parámetro la ubicación del archivo (`ruta`) y utiliza el método `pd.read_csv()` para cargar el contenido del dataset en un DataFrame. La estructura `try-except` implementa un mecanismo de control de errores. Primero intenta leer el archivo especificado; si el archivo no existe o la ruta es incorrecta, se genera una excepción `FileNotFoundError` acompañada de un mensaje descriptivo que orienta al usuario sobre cómo resolver el problema.

Una vez completada la lectura, la función utiliza el atributo `shape` para mostrar las dimensiones del conjunto de datos. El valor `df.shape[0]` representa la cantidad de filas, mientras que `df.shape[1]` indica el número de columnas.

### Carga del conjunto de datos

La función `cargar_datos()` lee el archivo `diabetes_raw.csv`, verifica su existencia y carga la información en un DataFrame de Pandas. Una vez completada la lectura, se muestran las dimensiones del conjunto de datos como una validación inicial del proceso de carga.

**Regla fundamental del pipeline:** el archivo crudo **nunca se modifica**. Todas las transformaciones se realizan sobre copias explícitas.

In [ ]:
df_raw = cargar_datos(RUTA_RAW)

print("=" * 55)
print("CARGA DEL DATASET")
print("=" * 55)
print(f"Archivo     : {RUTA_RAW}")
print(f"Dimensiones : {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print("\nColumnas y tipos:")
for col in df_raw.columns:
    print(f"  - {col:<28} [{df_raw[col].dtype}]")
print("\nPrimeras 5 filas:")
print(df_raw.head().to_string(index=False))

### Visualización inicial del conjunto de datos

Una vez cargado el dataset, es recomendable visualizar las primeras filas para verificar que la lectura se realizó correctamente. Esta inspección preliminar permite confirmar que los nombres de las columnas son los esperados, que los datos se encuentran correctamente delimitados y que no existen problemas evidentes durante el proceso de importación.

Para ello se utiliza el método `head()`, que muestra por defecto los primeros cinco registros del DataFrame. Esta revisión constituye una validación rápida de la estructura general de los datos antes de iniciar etapas más detalladas de exploración, limpieza y transformación.

In [ ]:
df_raw.head()

## 2. Exploración Inicial del Conjunto de Datos

### ¿Qué se realiza en esta etapa?

Antes de aplicar cualquier proceso de limpieza o transformación, es fundamental conocer el estado original del conjunto de datos. La exploración inicial permite identificar posibles problemas de calidad de datos, comprender la estructura del dataset y establecer una base sólida para las decisiones de preprocesamiento.

### Información analizada

La función `explorar_dataframe()` realiza un análisis preliminar del conjunto de datos y reporta los siguientes aspectos:

#### Dimensiones del dataset
Mediante el atributo `shape` se obtiene la cantidad de filas y columnas presentes en el DataFrame. Esta información permite verificar que el archivo fue leído correctamente.

#### Tipos de datos
A través del atributo `dtypes` se identifican los tipos de datos asociados a cada variable. Este análisis permite distinguir variables numéricas de variables de texto y detectar posibles inconsistencias en el almacenamiento de los datos.

#### Valores faltantes
Utilizando `isnull().sum()`, se calcula la cantidad de valores nulos presentes en cada columna. La detección temprana de valores faltantes es fundamental para decidir la estrategia de imputación o eliminación adecuada en etapas posteriores.

#### Estadísticos descriptivos
La función `describe()` genera un resumen estadístico de las variables numéricas, incluyendo medidas como cantidad de registros, media, desviación estándar, mínimo, percentiles y máximo.

### Importancia de la exploración inicial

Realizar una exploración preliminar del dataset es una práctica esencial en cualquier proyecto de Ciencia de Datos. Esta etapa proporciona una visión general del estado de los datos, facilita la identificación de problemas de calidad y permite fundamentar las decisiones de limpieza y transformación en evidencia objetiva.

In [ ]:
# Alias para compatibilidad con el nombre del original
explorar_dataframe = explorar_dataset

explorar_dataset(df_raw)

### Visualización de distribuciones — Dataset crudo

Los histogramas permiten identificar visualmente la distribución de cada variable, detectar asimetrías y confirmar los hallazgos numéricos del EDA. Se muestran las distribuciones **antes de cualquier transformación** para tener una línea base de comparación con el dataset procesado.

In [ ]:
df_raw.select_dtypes(include='number').hist(
    bins=30, figsize=(14, 10), edgecolor='white', color='steelblue'
)
plt.suptitle('Distribuciones RAW — antes de limpieza', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Resumen de calidad y estructura de las variables

Antes de iniciar las transformaciones del conjunto de datos, es importante realizar una revisión general de cada una de las variables disponibles. Para cada columna se examinan los siguientes aspectos:

- **Nombre de la variable:** permite identificar el atributo evaluado.
- **Tipo de dato:** indica cómo Pandas interpreta los valores almacenados en cada columna.
- **Cantidad de valores nulos:** permite identificar variables con información incompleta.
- **Cantidad de valores únicos:** facilita la detección de posibles inconsistencias.

Esta revisión proporciona una visión general del estado del dataset y constituye una herramienta de diagnóstico útil para las etapas siguientes del pipeline de preprocesamiento.

In [ ]:
for col in df_raw.columns:
    print(f"\n{'='*50}")
    print(f"Columna: {col}")
    print(f"Tipo: {df_raw[col].dtype}")
    print(f"Nulos: {df_raw[col].isnull().sum()}")
    print(f"Valores únicos: {df_raw[col].nunique()}")

### Revisión de categorías y detección de inconsistencias

Durante la exploración inicial del conjunto de datos, es posible encontrar variables que aparecen clasificadas como tipo texto (`object`) cuando en realidad deberían contener valores numéricos. Esta situación puede originarse por:

- Valores ingresados con formatos incorrectos.
- Caracteres alfabéticos en campos numéricos.
- Símbolos o caracteres especiales mezclados con datos numéricos.

Para identificar este tipo de situaciones, se analizan las categorías únicas presentes en cada variable clasificada como texto. Esta revisión permite:

- Detectar inconsistencias que afectan la calidad de los datos.
- Identificar registros que requieren limpieza o corrección.
- Fundamentar decisiones de transformación en evidencia objetiva.

La revisión detallada de los valores únicos constituye una etapa fundamental dentro del proceso de preprocesamiento, ya que permite anticipar problemas que podrían afectar el análisis posterior.

In [ ]:
columnas_texto = df_raw.select_dtypes(include=['object', 'string']).columns
for col in columnas_texto:
    print(f"\n{'='*60}")
    print(f"ANÁLISIS DE: {col}")
    print(f"{'='*60}")
    print(df_raw[col].value_counts(dropna=False))

### Hallazgos del EDA — resumen de problemas detectados

| # | Problema | Columna(s) afectada(s) | Magnitud | Estrategia |
|---|---|---|---|---|
| 1 | Tipo incorrecto (`str`) | `BloodPressure` | 2 045 filas | Limpiar `'mmHg'` y `'?'` → `pd.to_numeric` |
| 2 | Ceros biológicamente imposibles | `Glucose`, `BloodPressure`, `BMI`, `Insulin` | 13–634 ceros | Reemplazar `0 → NaN` |
| 3 | Valor centinela `Glucose = 999` | `Glucose` | 12 filas | Reemplazar `> 300 → NaN` |
| 4 | Valor centinela `Age = 150` | `Age` | 15 filas | Reemplazar `> 100 → NaN` |
| 5 | Outlier extremo `BMI > 60` | `BMI` | 2 filas | Reemplazar `> 60 → NaN` |
| 6 | Valores nulos reales (NaN) | `SkinThickness` (8%), `Insulin` (12%) | 164 + 245 | Ver decisión en sección 3 |
| 7 | Filas duplicadas exactas | Dataset completo | 45 filas | `drop_duplicates` |

## 3. Limpieza de Datos

### ¿Qué se realiza en esta etapa?

La limpieza de datos tiene como objetivo mejorar la calidad y consistencia del conjunto antes de aplicar transformaciones y modelado. Se aplican cuatro funciones en secuencia, cada una con un **único propósito**. Todas operan sobre **copias del DataFrame** y jamás modifican el dataset crudo.

### Análisis de la variable Insulin mediante diagrama de caja

Con el objetivo de comprender mejor el comportamiento de la variable `Insulin`, se generó un diagrama de caja (*boxplot*), herramienta que permite visualizar la distribución de los datos, identificar valores atípicos y evaluar la presencia de posibles anomalías en la variable.

El gráfico evidencia una distribución asimétrica con una marcada concentración de observaciones en valores bajos y una cola extensa hacia valores altos. Este comportamiento indica que la variable presenta una distribución sesgada hacia la derecha, característica habitual en variables biomédicas relacionadas con mediciones fisiológicas.

Asimismo, se observa la presencia de múltiples valores atípicos ubicados sobre el límite superior del diagrama. Sin embargo, estos registros forman parte de la variabilidad natural de la variable y no constituyen necesariamente errores de medición, por lo que no se justifica su eliminación automática.

Durante la exploración inicial también se identificaron valores faltantes en `Insulin`. Considerando que la variable contiene información potencialmente relevante para el análisis de diabetes, se decidió conservarla dentro del conjunto de datos imputando los valores faltantes con la mediana.

La estrategia adoptada busca preservar la calidad y representatividad de los datos, manteniendo una variable clínicamente significativa dentro del conjunto de datos y minimizando el riesgo de introducir sesgos derivados de procesos de imputación.

In [ ]:
# Boxplot de Insulin — línea base antes de la limpieza
fig, ax = plt.subplots(figsize=(6, 4))
df_raw['Insulin'].dropna().plot.box(ax=ax)
ax.set_title('Distribución de Insulin — Dataset crudo')
ax.set_ylabel('mu U/ml')
plt.tight_layout()
plt.show()

### Eliminación de registros con valores faltantes — criterio para Insulin

Durante la exploración inicial se detectó la presencia de valores faltantes en la variable `Insulin`. Se optó por imputar los valores faltantes con la mediana en lugar de eliminar los registros, preservando así la información disponible.

### Eliminación de la variable SkinThickness

La variable `SkinThickness` fue analizada durante la exploración inicial. Presenta 504 ceros (datos faltantes codificados) más 164 NaN reales, totalizando 668 de 2 045 registros sin información confiable (32.6%). Mantener atributos con información limitada o de baja calidad puede afectar negativamente los análisis posteriores. Por esta razón, se decidió excluir esta variable antes de continuar con las etapas de transformación y preparación de los datos.

### Detección de texto en BloodPressure

Durante la revisión de los valores únicos de la variable `BloodPressure`, se identificó la presencia de registros almacenados con formatos inconsistentes. Aunque la mayoría de los valores corresponden a mediciones numéricas de presión arterial diastólica, algunos registros incluyen texto adicional asociado a la unidad de medida (p. ej. `'72 mmHg'`) y 20 registros contienen el símbolo `'?'` indicando valor desconocido.

Esta situación genera un problema de calidad de datos, ya que una misma variable contiene simultáneamente valores numéricos y valores de texto. Mantener este tipo de inconsistencias puede afectar el análisis posterior, dificultando la realización de cálculos estadísticos y el entrenamiento de modelos predictivos.

Por esta razón, antes de continuar con las etapas de transformación y modelado, resulta necesario identificar y corregir estas inconsistencias:

1. **Eliminar las unidades de medida** presentes en los registros inconsistentes.
2. **Convertir la columna a un tipo de dato numérico** mediante `pd.to_numeric(errors='coerce')`.

Al finalizar la corrección, todos los registros de la variable `BloodPressure` estarán almacenados de manera homogénea y correctamente tipados para su uso en análisis estadísticos y modelado.

In [ ]:
# ── Ejecutar pipeline de limpieza ────────────────────────────
df = df_raw.copy()          # copia de trabajo — raw nunca se toca
df_original = df.copy()     # referencia para comparación final

df = eliminar_columnas(df, ['SkinThickness'])
df = castear_bloodpressure(df)
df = reemplazar_invalidos(df)
df = eliminar_duplicados(df)

print(f"\n{'='*55}")
print("RESUMEN POST-LIMPIEZA INICIAL")
print(f"{'='*55}")
print(f"  Filas    : {df.shape[0]}")
print(f"  Columnas : {df.shape[1]}")
print(f"  NaN total: {df.isna().sum().sum()}")
print(f"\nNaN por columna:")
print(df.isna().sum().to_string())

## 4. Imputación de Valores Faltantes

### ¿Por qué imputar con la mediana?

Las variables con valores faltantes (`Glucose`, `BloodPressure`, `BMI`, `Insulin`, `Age`) presentan distribuciones asimétricas positivas con colas largas, característica habitual en variables biomédicas. Debido a que las variables analizadas presentan distribuciones asimétricas y contienen valores atípicos, la mediana constituye una medida más robusta que la media para representar el valor central de la distribución.

La decisión de imputar en lugar de eliminar responde a preservar la mayor cantidad de información disponible, evitando reducir el tamaño del dataset de manera innecesaria. La mediana se calcula **después** de haber reemplazado los inválidos, por lo que refleja únicamente los datos clínicamente válidos.

In [ ]:
COLS_IMPUTAR = ['Glucose', 'BloodPressure', 'BMI', 'Insulin', 'Age']

df = imputar_mediana(df, COLS_IMPUTAR)

## 5. Validación Posterior a la Limpieza

### ¿Por qué realizar una nueva validación?

Una vez aplicadas las transformaciones de limpieza, es necesario verificar nuevamente la calidad del conjunto de datos. Este proceso permite confirmar que las acciones realizadas produjeron los resultados esperados y que el dataset se encuentra en condiciones adecuadas para continuar con las etapas de transformación y preparación de variables.

La validación posterior a la limpieza tiene como objetivo identificar posibles inconsistencias remanentes, verificar la ausencia de valores faltantes y confirmar que los tipos de datos sean coherentes con la naturaleza de cada variable.

### Aspectos revisados

Durante esta etapa se realiza una nueva inspección de cada columna considerando:

- **Tipo de dato** asignado por Pandas.
- **Cantidad de valores faltantes.**
- **Cantidad de valores únicos:** permite identificar posibles valores inesperados o inconsistencias residuales.
- **Coherencia** de las variables con respecto a su dominio de datos.

Esta revisión permite comprobar que las modificaciones realizadas durante la limpieza fueron efectivas y proporciona una base confiable para continuar con las siguientes fases del pipeline de preprocesamiento.

### Resumen de calidad de datos posterior a la limpieza

Tras aplicar las correcciones necesarias sobre las variables que presentaban inconsistencias, se realiza una nueva evaluación del conjunto de datos con el propósito de verificar el resultado de las transformaciones efectuadas.

La revisión evidencia que las variables previamente afectadas por inconsistencias han sido corregidas exitosamente y que los atributos numéricos se encuentran almacenados utilizando tipos de datos apropiados para su procesamiento. Como resultado, el conjunto de datos presenta una estructura más consistente, homogénea y preparada para las etapas posteriores del proyecto.

In [ ]:
validar_post_limpieza(df)

## 6. Validación de Rangos y Análisis de Outliers

### ¿Por qué validar los rangos de los datos?

Aunque las variables ya se encuentran almacenadas utilizando tipos de datos numéricos adecuados, aún es posible que existan valores que no sean coherentes con el dominio del problema. Estos registros pueden originarse por errores de digitación, problemas durante la captura de datos o inconsistencias presentes en la fuente original.

Por esta razón, se realiza una validación de rangos sobre las variables numéricas más relevantes del conjunto de datos.

### Variables evaluadas

| Variable | Restricción esperada |
|-----------|---------------------|
| Pregnancies | ≥ 0 |
| Glucose | > 0 |
| BloodPressure | > 0 |
| Insulin | > 0 |
| BMI | > 0 |
| DiabetesPedigreeFunction | > 0 |
| Age | > 0 |

La detección de valores fuera de estos rangos permite identificar posibles inconsistencias que podrían afectar la calidad del análisis y del modelado posterior.

### Análisis de valores iguales a cero en la variable Insulin

Durante la validación de rangos se identificó una cantidad considerable de registros con valor igual a cero en la variable `Insulin`. Este hallazgo requiere una revisión específica, ya que la interpretación de estos valores puede tener un impacto significativo en la calidad del conjunto de datos.

La variable `Insulin` representa el nivel de insulina sérica medido en unidades fisiológicas. Desde una perspectiva clínica, un valor de insulina igual a cero es poco frecuente y, en la mayoría de los casos, suele asociarse a mediciones no registradas, datos faltantes o valores utilizados como marcador para indicar ausencia de información.

Por esta razón, los valores iguales a cero no deben interpretarse automáticamente como mediciones válidas, sino que requieren un análisis adicional para determinar si representan observaciones reales o posibles inconsistencias en el proceso de captura de datos.

La importancia de esta revisión radica en que una alta proporción de valores iguales a cero puede afectar la distribución de la variable, alterar medidas estadísticas como la media y la mediana, e influir en el comportamiento de los modelos predictivos. Por ello, antes de aplicar cualquier técnica de transformación o modelado, resulta necesario cuantificar la cantidad de registros con valor cero y evaluar su impacto sobre la calidad de la información disponible.

La evaluación de estos valores constituye una etapa fundamental dentro del proceso de limpieza de datos, ya que contribuye a garantizar que las decisiones tomadas durante el preprocesamiento estén respaldadas por evidencia objetiva y criterios metodológicos consistentes.

### Tratamiento de valores inconsistentes — estrategia aplicada

1. **Cuantificar la magnitud del problema:** se calcula la cantidad y el porcentaje de registros con valores iguales a cero para cada variable afectada.

2. **Evaluar la naturaleza de los valores detectados:** en variables como `Glucose`, `BloodPressure` y `BMI`, un valor igual a cero no resulta fisiológicamente posible, por lo que dichos registros serán considerados como valores faltantes implícitos.

3. **Analizar específicamente la variable Insulin:** debido a la elevada cantidad de registros con valor cero observada durante la exploración inicial, se realiza una evaluación particular de esta variable para determinar si su calidad permite conservarla dentro del conjunto de datos.

4. **Transformar los valores inconsistentes:** los valores identificados como inconsistentes serán convertidos a `NaN` para facilitar su posterior tratamiento mediante técnicas de imputación.

5. **Validar nuevamente el resultado:** se verifica que las inconsistencias hayan sido correctamente gestionadas.

La aplicación de este procedimiento permitirá disponer de un conjunto de datos más consistente y representativo, reduciendo el riesgo de introducir sesgos o errores durante las etapas posteriores de análisis y modelado predictivo.

In [ ]:
variables_numericas = ['Pregnancies', 'Glucose', 'BloodPressure',
                       'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

print("VALIDACIÓN DE RANGOS")
print(analizar_rangos(df, variables_numericas).to_string(index=False))

print("\nOUTLIERS ESTADÍSTICOS (método IQR)")
print(detectar_outliers_iqr(df, variables_numericas + ['Outcome']).to_string(index=False))

### Visualización de la distribución de las variables — post-limpieza

A continuación, se genera un diagrama de caja para las variables con mayor concentración de outliers. El objetivo de este análisis no es eliminar automáticamente los valores detectados, sino comprender la distribución de las variables e identificar posibles casos extremos que requieran atención especial.

Los resultados obtenidos servirán como base para complementar el análisis estadístico utilizando el método IQR.

In [ ]:
# Visualización de la variable con mayor concentración de outliers
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

df.boxplot(column='Insulin', ax=axes[0])
axes[0].set_title('Distribución de Insulin (post-limpieza)')
axes[0].set_ylabel('mu U/ml')

df.boxplot(column='DiabetesPedigreeFunction', ax=axes[1])
axes[1].set_title('Distribución de DiabetesPedigreeFunction')
axes[1].set_ylabel('Valor')

plt.tight_layout()
plt.show()
print("Los outliers visibles corresponden a variabilidad clínica real y no se eliminan.")

### Criterios para la toma de decisiones sobre variables con alta proporción de ceros

Una vez identificada la magnitud de los ceros en cada variable, se aplican los siguientes umbrales para decidir la estrategia de tratamiento:

| Porcentaje de valores cero | Interpretación | Acción aplicada |
|---|---|---|
| < 20% | Impacto bajo | Reemplazar por NaN e imputar mediana |
| 20% – 50% | Calidad moderada | Evaluar caso a caso; en este proyecto se imputa |
| 50% – 70% | Calidad reducida | Evaluar utilidad de la variable |
| > 70% | Calidad insuficiente | Eliminar la variable |

**Aplicación al dataset:**

- `Glucose` (0.63% ceros): impacto mínimo → imputar mediana.
- `BloodPressure` (3.50% ceros): impacto bajo → imputar mediana.
- `BMI` (0.89% ceros): impacto mínimo → imputar mediana.
- `Insulin` (35.22% ceros): calidad moderada → se imputa con mediana dado que la variable es clínicamente relevante para la predicción de diabetes.
- `SkinThickness` (32.6% entre ceros + NaN reales): umbral limítrofe + alta proporción de NaN reales → **se elimina la columna completa** para no introducir sesgo por imputación masiva.

### Comparación de inconsistencias entre variables numéricas

Para complementar el análisis, se realiza una evaluación conjunta de todas las variables numéricas del conjunto de datos. El objetivo es identificar cuáles presentan la mayor cantidad de valores potencialmente inconsistentes.

La siguiente tabla resume la cantidad y el porcentaje de registros con valores menores o iguales a cero para cada variable analizada. Esta información permite comparar la calidad relativa de las variables y proporciona evidencia objetiva para justificar las decisiones de limpieza que serán aplicadas posteriormente.

### Visualización de distribuciones — Dataset post-limpieza

Comparar las distribuciones antes y después de la limpieza permite verificar visualmente que las transformaciones aplicadas no distorsionaron las distribuciones originales de las variables clínicas.

In [ ]:
df[df.select_dtypes(include='number').columns].hist(
    bins=30, figsize=(14, 10), edgecolor='white', color='darkorange'
)
plt.suptitle('Distribuciones POST-LIMPIEZA — después de imputación', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print("Comparación: las distribuciones se mantienen similares al raw, confirmando que")
print("la imputación por mediana no distorsionó los datos.")

### ¿Qué son los valores atípicos?

Los valores atípicos (*outliers*) son observaciones que se encuentran significativamente alejadas del comportamiento general del conjunto de datos. Su presencia puede deberse a errores de medición, errores de captura, o bien a observaciones legítimas que representan casos extremos pero reales dentro de la población estudiada.

La identificación de valores atípicos es una etapa importante dentro del preprocesamiento de datos, ya que su presencia puede afectar el entrenamiento de modelos predictivos y distorsionar las medidas estadísticas.

## 7. Detección y Análisis de Valores Atípicos

### Método IQR

Para evaluar la presencia de valores atípicos se utiliza el método del rango intercuartílico (IQR), ampliamente utilizado en análisis exploratorio de datos:

- Primer cuartil (Q1) y tercer cuartil (Q3).
- Rango intercuartílico (IQR = Q3 − Q1).
- Límite inferior = Q1 − 1.5 × IQR.
- Límite superior = Q3 + 1.5 × IQR.
- Cualquier observación fuera de estos límites se considera un posible valor atípico.

Este análisis permite cuantificar objetivamente la cantidad de registros extremos presentes en cada variable numérica.

### Interpretación de los resultados

La tabla generada muestra la cantidad de observaciones identificadas como valores atípicos para cada variable numérica.

Es importante señalar que la detección de un valor atípico no implica necesariamente que dicho registro sea incorrecto o deba eliminarse. En conjuntos de datos clínicos y biomédicos es habitual encontrar observaciones extremas que representan condiciones reales de algunos pacientes.

Por esta razón, los resultados obtenidos deben interpretarse considerando el contexto del problema y el significado de cada variable, evitando eliminar información potencialmente relevante sin una justificación técnica adecuada.

In [ ]:
# Diagrama de caja para cada variable numérica — detección visual de outliers
columnas_numericas = df.select_dtypes(include=['int64', 'float64']).columns

for col in columnas_numericas:
    plt.figure(figsize=(8, 2))
    plt.boxplot(df[col], vert=False)
    plt.title(f'Boxplot — {col}')
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

## 7. Transformaciones y Variables Derivadas

Con el dataset limpio y sin valores faltantes, se generan variables derivadas que aportan valor analítico adicional:

| Variable derivada | Basada en | Justificación |
|---|---|---|
| `AgeGroup` | `Age` | Rangos etarios estándar en epidemiología de DM2: facilita análisis por subgrupo clínico |
| `BMI_Category` | `BMI` | Clasificación OMS (Bajo peso, Normal, Sobrepeso, Obesidad): mejora la interpretabilidad del modelo |

In [ ]:
df = crear_variables_derivadas(df)

### Codificación de variables categóricas (One-Hot Encoding)

Las variables `AgeGroup` y `BMI_Category` son categóricas y no pueden usarse directamente en la mayoría de los algoritmos de aprendizaje automático, que requieren entradas numéricas. Se aplica **One-Hot Encoding** mediante `pd.get_dummies()`:

- Convierte cada categoría en una columna binaria (0/1).
- `drop_first=True` elimina una categoría por variable para evitar la **trampa de la variable dummy** (multicolinealidad perfecta en modelos lineales).
- `dtype=int` asegura que las columnas resultantes sean enteras, no booleanas.

In [ ]:
df = codificar_categoricas(df, ['AgeGroup', 'BMI_Category'])

print(f'\nDataset tras OHE: {df.shape[0]} filas × {df.shape[1]} columnas')
print('Columnas con dummies:')
dummy_cols = [c for c in df.columns if 'AgeGroup_' in c or 'BMI_Category_' in c]
for c in dummy_cols:
    print(f'  - {c}  [distribución: {df[c].sum()} unos de {len(df)} filas]')

## 8. Escalamiento de Variables

### ¿Por qué es necesario escalar los datos?

Las variables del conjunto de datos se encuentran expresadas en diferentes unidades de medida y presentan rangos de valores considerablemente distintos. Por ejemplo:

- La edad se expresa en años.
- La glucosa corresponde a una concentración plasmática en mg/dL.
- El índice de masa corporal utiliza una escala diferente.
- La función de predisposición genética posee valores mucho menores que otras variables.

Estas diferencias pueden provocar que algunas variables tengan una influencia desproporcionada sobre determinados algoritmos de aprendizaje automático.

### Estandarización mediante `StandardScaler`

La técnica `StandardScaler` transforma cada variable numérica para que posea:

- Media igual a 0.
- Desviación estándar igual a 1.

Esta transformación conserva la información contenida en los datos, pero homogeniza las escalas de todas las variables, facilitando su comparación y mejorando el comportamiento de numerosos algoritmos de aprendizaje automático.

Se elige `StandardScaler` sobre `MinMaxScaler` porque:
1. Las variables clínicas presentan distribuciones asimétricas con colas largas. `MinMaxScaler` sería sensible a los outliers residuales.
2. `StandardScaler` maneja mejor estas distribuciones al no depender de valores extremos.
3. Es la opción estándar para algoritmos como SVM, regresión logística y PCA.

In [ ]:
columnas_escalar = ['Pregnancies', 'Glucose', 'BloodPressure',
                    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

df = escalar_variables(df, columnas_escalar)

In [ ]:
# Validación del escalamiento: media ≈ 0, std ≈ 1
df[columnas_escalar].describe().T[['mean', 'std']]

### Validación del escalamiento

Una vez aplicada la transformación mediante `StandardScaler`, se verificaron las estadísticas descriptivas de las variables numéricas escaladas.

Los resultados muestran que todas las variables presentan una media muy cercana a cero y una desviación estándar cercana a uno, comportamiento esperado cuando la estandarización ha sido aplicada correctamente.

Las pequeñas diferencias observadas respecto a los valores exactos de 0 y 1 corresponden a errores de precisión numérica inherentes a los cálculos realizados por el computador y no representan un problema para el análisis.

Estos resultados confirman que las variables fueron normalizadas exitosamente, quedando expresadas en una escala común que facilita su utilización en algoritmos de aprendizaje automático y análisis estadísticos posteriores.

## 9. Validación Técnica Final

La función `validar_dataset_final()` verifica sistemáticamente los supuestos críticos que debe cumplir el dataset antes del modelado. Cada `assert` dispara un `AssertionError` con mensaje descriptivo si el supuesto falla, garantizando que ningún error pase silenciosamente.

Se verifican casos normales, casos límite y posibles excepciones del pipeline.

In [ ]:
validar_dataset_final(df_raw, df)

## 10. Comparación Antes / Después del Pipeline

In [ ]:
resumen_comparativo(df_raw, df)

## 11. Exportación del Conjunto de Datos Procesado

Una vez completadas las actividades de exploración, limpieza, validación, tratamiento de valores faltantes, escalamiento y generación de variables derivadas, se genera una versión final del conjunto de datos preparada para las etapas posteriores del proyecto.

La exportación del dataset permite conservar todas las transformaciones realizadas durante esta fase y garantiza la disponibilidad de una fuente de datos consistente para los análisis y modelos que serán desarrollados en las siguientes etapas.

El archivo será almacenado en la misma carpeta del notebook siguiendo la estructura definida para el proyecto.

In [ ]:
exportar_dataset(df, RUTA_PROCESSED)

## 10. Resumen de la Fase

Durante esta fase se desarrolló un proceso completo de preparación de datos sobre el conjunto de datos de diabetes.

Las principales actividades realizadas fueron:

- Obtención y carga de los datos.
- Exploración inicial de la estructura del dataset.
- Identificación de tipos de datos y valores faltantes.
- Detección y corrección de inconsistencias (ceros, centinelas, tipos incorrectos).
- Eliminación de 45 filas duplicadas exactas.
- Análisis de rangos válidos para variables numéricas.
- Tratamiento de valores faltantes mediante imputación por mediana.
- Detección y análisis de valores atípicos mediante método IQR.
- Generación de variables derivadas (AgeGroup, BMI_Category).
- Codificación de variables categóricas con One-Hot Encoding.
- Estandarización de variables numéricas mediante `StandardScaler`.
- Validación técnica con `assert` sobre 7 supuestos críticos.
- Generación y almacenamiento del dataset procesado.

Como resultado, se obtuvo un conjunto de datos **limpio, consistente y preparado** para las etapas posteriores de análisis exploratorio avanzado y modelado predictivo.

---
### Consideraciones sobre la organización del código

Las funciones del pipeline fueron centralizadas en el archivo `src/utils.py`, el cual se importa al inicio del notebook mediante:

```python
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.utils import *
```

Este enfoque garantiza que todas las funciones estén disponibles desde cualquier celda del notebook, haciéndolas verdaderamente reutilizables e independientes del orden de ejecución. Mantener las funciones en un módulo externo mejora la modularidad, facilita el mantenimiento y sigue las buenas prácticas de desarrollo de software en proyectos de ciencia de datos.

La estructura del repositorio refleja esta organización:

```
F2/
├── notebooks/
│   └── F2_Preprocesamiento.ipynb   ← invoca funciones desde src/utils
├── src/
│   └── utils.py                    ← centraliza las 16 funciones
├── data/
│   ├── raw/
│   │   └── diabetes_raw.csv
│   └── processed/
│       └── diabetes_processed.csv
└── README.md
```

---
*Notebook ejecutado con `Kernel → Restart & Run All` — sin errores — numeración continua.*